## Understanding & Cleaning Data

In [ ]:
from function import CleaningData
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_Trans = pd.read_parquet('transactions.parquet')
df_TransItem = pd.read_parquet('transaction_items.parquet')
df_Users = pd.read_parquet('users.parquet')
df_stores = pd.read_parquet('stores.parquet')
df_menu = pd.read_parquet('menu_items.parquet')
df_vouchers = pd.read_parquet('vouchers.parquet')
df_payment = pd.read_parquet('paymentsUnique.parquet')

# Katalog Data

## df Trans

In [ ]:
df_Trans.describe(include='all')

## df_TransItem

In [ ]:
df_TransItem.describe(include="all")

## df_Users

In [ ]:
df_Users.describe(include='all')

## df stores

In [ ]:
df_stores.describe(include="all")

## df Menu

In [ ]:
df_menu.describe(include='all')

# Invalid Values

## df Trans

In [ ]:
orphans = df_Trans[
    (~df_Trans['user_id'].isin(df_Users['user_id'])) & 
    (df_Trans['user_id'].notna())
]

print(f"Jumlah Orphan Transactions (ID Siluman): {len(orphans)}")
if len(orphans) > 0:
    display(orphans.head(10))
    print(f"User ID yang bermasalah: {orphans['user_id'].unique()}")

In [ ]:
minus_trans = df_Trans[
    (df_Trans['original_amount'] < 0) | 
    (df_Trans['final_amount'] < 0) | 
    (df_Trans['discount_applied'] < 0)
]
print(f"Transaksi dengan nominal minus di df_Trans: {len(minus_trans)} baris")

In [ ]:
df_Trans['Non-Member'] = df_Trans['user_id'].isna()
df_Trans['Member'] = df_Trans['user_id'].notna()

In [ ]:
member = df_Trans.index[df_Trans['Member'] == 1]
non_member = df_Trans.index[df_Trans['Non-Member'] == 1]

In [ ]:
print(f"Jumlah transaksi member: {len(member)}")
print(f"Jumlah transaksi non-member: {len(non_member)}")
print(f"Total transaksi dari unknown user: {len(member) - len(non_member)}")

## df Trans Items

In [ ]:
minus_item = df_TransItem[
    (df_TransItem['quantity'] <= 0) | 
    (df_TransItem['unit_price'] < 0) | 
    (df_TransItem['subtotal'] < 0)
]
print(f"Item dengan qty/harga minus di df_TransItem: {len(minus_item)} baris\n")

In [ ]:
orphan_items = df_TransItem[~df_TransItem['transaction_id'].isin(df_Trans['transaction_id'])]
print(f"Item siluman (nggak punya induk struk transaksi): {len(orphan_items)} baris")

In [ ]:
unregistered_menu_sold = df_TransItem[~df_TransItem['item_id'].isin(df_menu['item_id'])]
print(f"Menu terjual yang belum terdaftar di Master Menu: {len(unregistered_menu_sold)} baris")

In [ ]:
price_consistency = df_TransItem.groupby('item_id')['unit_price'].nunique()
bad_items = price_consistency[price_consistency > 1]
print(f"Jumlah Menu yang harganya tidak konsisten: {len(bad_items)}")

## df Users

In [ ]:
today = pd.Timestamp.now()
print(len(df_Users['birthdate']))
print(len(df_Users['registered_at']))
df_Users = df_Users[
    (df_Users['birthdate'] < today) &
    (df_Users['registered_at'] < today)
]
print(len(df_Users['birthdate']))
print(len(df_Users['registered_at']))

# Handling Duplikat

## df Trans

In [ ]:
df_Trans.duplicated().sum()

## df Trans Items

In [ ]:
df_TransItem.duplicated().sum()

In [ ]:
print(f'Data duplikat dari transaction id: {len(df_TransItem[df_TransItem["transaction_id"].duplicated(keep=False)])}')
print(f'Data duplikat dari item id: {len(df_TransItem[df_TransItem["item_id"].duplicated(keep=False)])}')
print(f'Data duplikat dari created at: {len(df_TransItem[df_TransItem["created_at"].duplicated(keep=False)])}')
print(f'Data duplikat dari transaction id, item id, dan created at: {len(df_TransItem[df_TransItem.duplicated(subset=["transaction_id", "item_id", "created_at"], keep=False)])}')

In [ ]:
mask_duplikat = df_TransItem.duplicated(
    subset=['transaction_id', 'item_id', 'created_at'],
    keep=False
)

df_TIDuplikat = (
    df_TransItem[mask_duplikat]
    .sort_values(
        by=[
            'transaction_id',
            'item_id',
            'created_at'
        ]
    )
)

print(f"""
Ditemukan {len(df_TIDuplikat)} data duplikat
""")
df_TIDuplikat['time_diff'] = (
    df_TIDuplikat
    .groupby(
        ['transaction_id', 'item_id']
    )['created_at']
    .diff()
    .dt.total_seconds()
)
zero_sec = (
    df_TIDuplikat['time_diff'] == 0
)

sec = (
    df_TIDuplikat['time_diff'] <= 30
)

print(f"""
Transaksi Duplikat dengan selisih waktu 0 detik:
{zero_sec.sum()}
""")
print(f"""Transaksi Duplikat dengan selisih waktu 30 detik:
{sec.sum()} """)

In [ ]:
index_sampah = df_TIDuplikat[sec].index
index_sampah
df_TransItem = df_TransItem[~df_TransItem.index.isin(index_sampah)]
del index_sampah
del df_TIDuplikat

print(f'Data duplikat dari transaction id, item id, dan created at: {len(df_TransItem[df_TransItem.duplicated(subset=["transaction_id", "item_id", "created_at"], keep=False)])}')

In [ ]:
df_TransItem.duplicated().sum()

## df Users

In [ ]:
df_Users.duplicated().sum()

# Handling Misscalculation

## df Trans

In [ ]:
inconsistent_trans = df_Trans[
    df_Trans['original_amount'] - df_Trans['discount_applied'] != df_Trans['final_amount']
]
print(f"Data ngaco: {len(inconsistent_trans)} baris\n")

In [ ]:
df_Trans = df_Trans.assign(final_amount = lambda x: x['original_amount'] - x['discount_applied'])
inconsistent_trans = df_Trans[
    df_Trans['original_amount'] - df_Trans['discount_applied'] != df_Trans['final_amount']
]
print(f"Data ngaco: {len(inconsistent_trans)} baris\n")

In [ ]:
df_Trans = df_Trans.merge(df_vouchers, on='voucher_id', how='left')
conditions = [
    (df_Trans['discount_type'] == 'percentage'),
    (df_Trans['discount_type'] == 'fixed')
]
choices = [
    (df_Trans['original_amount'] * df_Trans['discount_value'] / 100),
    df_Trans['discount_value']
]

df_Trans['calculated_discount'] = np.select(conditions, choices, default=0)
mismatch_discount = df_Trans[
    (df_Trans['voucher_id'].notna()) & 
    (np.abs(df_Trans['discount_applied'] - df_Trans['calculated_discount']) > 0.01)
]

print(f"Transaksi dengan diskon gak masuk akal: {len(mismatch_discount)}")

In [ ]:
df_Trans['created_at'] = pd.to_datetime(df_Trans['created_at'])
expired_usage = df_Trans[
    (df_Trans['voucher_id'].notna()) & 
    ((df_Trans['created_at'] < df_Trans['valid_from']) | 
     (df_Trans['created_at'] > df_Trans['valid_to']))
]

print(f"Jumlah transaksi pake voucher kadaluarsa/pre-mature: {len(expired_usage)}")

In [ ]:
expired_usage['days_diff'] = np.where(
    expired_usage['created_at'] < expired_usage['valid_from'],
    (expired_usage['valid_from'] - expired_usage['created_at']).dt.days,
    (expired_usage['created_at'] - expired_usage['valid_to']).dt.days
)

print("Distribusi penyimpangan hari:")
print(expired_usage['days_diff'].value_counts().head(10))

In [ ]:
trans_date = df_Trans['created_at'].dt.normalize()
valid_from = df_Trans['valid_from'].dt.normalize()
valid_to = df_Trans['valid_to'].dt.normalize()
real_expired = df_Trans[
    (df_Trans['voucher_id'].notna()) & 
    ((trans_date < valid_from) | (trans_date > valid_to))
]

print(f"Jumlah transaksi expired yang SEBENARNYA: {len(real_expired)}")

In [ ]:
df_Trans['final_amount'] = df_Trans['original_amount'] - df_Trans['discount_applied']

In [ ]:
minus_check = df_Trans[df_Trans['final_amount'] < 0]
print(f"Jumlah transaksi minus setelah sinkronisasi diskon: {len(minus_check)}")

## df Trans Items

In [ ]:
calc_subtotal = (
    df_TransItem['quantity'] *
    df_TransItem['unit_price']
)

bad_subtotal = df_TransItem[
    np.abs(calc_subtotal - df_TransItem['subtotal']) > 0.01
]

print(len(bad_subtotal))

In [ ]:
# 1. Hitung total item per transaksi
item_totals = df_TransItem.groupby('transaction_id')['subtotal'].sum().reset_index()
item_totals.columns = ['transaction_id', 'sum_of_items']

# 2. Gabungkan dengan tabel Transaksi (Header)
check_integrity = df_Trans.merge(item_totals, on='transaction_id', how='left')

# 3. Cari selisihnya
check_integrity['diff'] = check_integrity['original_amount'] - check_integrity['sum_of_items']

# 4. Cari yang selisihnya tidak nol
mismatch_original = check_integrity[check_integrity['diff'].abs() > 0.1]

print(f"Jumlah transaksi yang gak sinkron: {len(mismatch_original)}")

In [ ]:
# 1. Hitung total item per transaksi (Source of Truth)
item_totals = df_TransItem.groupby('transaction_id')['subtotal'].sum().reset_index()
item_totals.columns = ['transaction_id', 'calculated_original_amount']

# 2. Merge ke df_Trans
df_Trans = df_Trans.merge(item_totals, on='transaction_id', how='left')

# 3. Tangani transaksi yang tidak punya item (fillna 0)
df_Trans['calculated_original_amount'] = df_Trans['calculated_original_amount'].fillna(0)

# 4. Update Original Amount & Final Amount
# Rumus: Final = Original (Baru) - Discount
df_Trans['original_amount'] = df_Trans['calculated_original_amount']
df_Trans['final_amount'] = df_Trans['original_amount'] - df_Trans['discount_applied']

df_Trans.drop(columns=['calculated_original_amount'], inplace=True)

item_totals = df_TransItem.groupby('transaction_id')['subtotal'].sum().reset_index()
item_totals.columns = ['transaction_id', 'sum_of_items']

check_integrity = df_Trans.merge(item_totals, on='transaction_id', how='left')

check_integrity['diff'] = check_integrity['original_amount'] - check_integrity['sum_of_items']

mismatch_original = check_integrity[check_integrity['diff'].abs() > 0.1]

print(f"Jumlah transaksi yang header vs detail-nya gak sinkron: {len(mismatch_original)}")

In [ ]:
df_Trans[df_Trans['final_amount'] < 0]

In [ ]:
df_Trans['discount_applied'] = np.where(
    df_Trans['discount_applied'] > df_Trans['original_amount'],
    df_Trans['original_amount'],
    df_Trans['discount_applied']
)
df_Trans['final_amount'] = df_Trans['original_amount'] - df_Trans['discount_applied']
negative_check = len(df_Trans[df_Trans['final_amount'] < 0])
print(f"Jumlah transaksi negatif setelah capping: {negative_check}")
df_Trans[df_Trans['final_amount'] < 0]

# Handling Duplicates

## df Trans

In [ ]:
df_Trans.duplicated().sum()

## df Trans Items

In [ ]:
df_TransItem.duplicated().sum()

# Handling Missing Values

## df Trans

In [ ]:
df_Trans.isnull().sum()

## df Trans Item

In [ ]:
df_TransItem.isnull().sum()

## df Users

In [ ]:
df_Users.isnull().sum()

## 

In [ ]:
df_menu = df_menu.drop(columns=['available_from', 'available_to'])

# Handling Outliers

## df Trans

In [ ]:
skew_dfTrans = df_Trans['final_amount'].skew()
print(f"Skewness sebelum transformasi: {skew_dfTrans}")
df_Trans[["final_amount"]].boxplot(vert=False);
plt.show()

In [ ]:
c= CleaningData(df_Trans)
print('metode IQR:')
_,_,iqr_Data = c.iqr('final_amount')
skew_iqr = iqr_Data.skew()
print(f'skewness setelah IQR: {skew_iqr:.2f}')
c.BoxPlot('final_amount', Target=iqr_Data)

print('metode Capping:')
_,_,capped_data = c.capping('final_amount')
skew_capped = capped_data.skew()
print(f'skewness setelah Capping: {skew_capped:.2f}')
c.BoxPlot('final_amount', Target=capped_data)

print('metode Z-Score:')
_,_,z_data = c.z_score_method('final_amount')
skew_z = z_data.skew()
print(f'skewness setelah Z-Score: {skew_z:.2f}')
c.BoxPlot('final_amount', Target=z_data)

print(f'''Jumlah data sebelum menangani outlier: {len(df_Trans)}\n 
      Jumlah data setelah IQR: {len(iqr_Data)}\nPersentase: {(1 - len(iqr_Data) / len(df_Trans)) * 100:.2f}%\n 
      Jumlah data setelah Capping: {len(capped_data)}\nPersentase: {(1 - len(capped_data) / len(df_Trans)) * 100:.2f}%\n 
      Jumlah data setelah Z-Score: {len(z_data)}\nPersentase: {(1 - len(z_data) / len(df_Trans)) * 100:.2f}%''')



In [ ]:
print('Persebaran transformasi Z-Score:')
c.HistPlot('final_amount', Target=z_data)
print('Persebaran transformasi IQR:')
c.HistPlot('final_amount', Target=iqr_Data)
print('Persebaran transformasi Capping:')
c.HistPlot('final_amount', Target=capped_data)

In [ ]:
sns.scatterplot(
    data = df_Trans['final_amount']
).set(title='Scatter Plot Final Amount (Original Data)')
plt.show()


In [ ]:
sns.scatterplot(
    data = capped_data
).set(title='Scatter Plot Final Amount (Capping Method)')
plt.show()


## df Trans Item

In [ ]:
CleaningData(df_TransItem).BoxPlot('quantity')
CleaningData(df_TransItem).BoxPlot('subtotal')

# pipeline

In [ ]:
TransCapping = df_Trans.copy()

TransCapping['final_amount']= capped_data

# Cleaned Data

In [ ]:
TransCapping.to_parquet('transactions_capping.parquet', index=False, compression='snappy')
df_TransItem.to_parquet('transaction_items_cleaned.parquet', index=False, compression='snappy')
df_Users.to_parquet('users_cleaned.parquet', index=False, compression='snappy')
df_menu.to_parquet('menu_cleaned.parquet', index=False, compression='snappy')
df_stores.to_parquet('stores_cleaned.parquet', index=False, compression='snappy')

In [ ]:
import gc
gc.collect()